# Human Activity Recognition (HAR): External Validation

This notebook reproduces the external validation experiments using a held-out test set

In [1]:
# Reproducible setup and imports
import os
import random
import logging

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Import LdaBoost from local package
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


## Dataset and preprocessing


In [ ]:
import re

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

train_df = pd.read_csv("Data/train.csv")
test_df = pd.read_csv("Data/test.csv", names=list(train_df.columns), header=0)

train_df = clean_column_names(train_df)
test_df = clean_column_names(test_df)

X = train_df.drop(columns="Activity")
y = train_df["Activity"]
X_test = test_df.drop(columns="Activity")
y_test = test_df["Activity"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_test_enc = le.transform(y_test)

features = list(X.columns)



## Baseline: GBM, PCA+GBM, LDA+GBM on external test


In [ ]:
# Common scaler
scaler = Pipeline([("scaler", StandardScaler())])

# 1) GBM baseline
base_preprocess = ColumnTransformer(
    remainder="passthrough",
    transformers=[("scale", scaler, features)],
)
base_gbm = Pipeline([
    ("preprocess", base_preprocess),
    ("gb", GradientBoostingClassifier(random_state=RANDOM_SEED)),
])
base_gbm.fit(X, y_enc)
y_pred = base_gbm.predict(X_test)
print(f"GBM — Test accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

# 2) PCA + GBM
pca_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[
        (
            "pca",
            Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=max(1, X.shape[1] // 16), random_state=RANDOM_SEED)),
            ]),
            features,
        )
    ],
)
pca_gbm = Pipeline([
    ("preprocess", pca_preprocess),
    ("gb", GradientBoostingClassifier(random_state=RANDOM_SEED)),
])
pca_gbm.fit(X, y_enc)
y_pred = pca_gbm.predict(X_test)
print(f"PCA+GBM — Test accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

# 3) LDA + GBM
lda_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[
        (
            "lda",
            Pipeline([
                ("scaler", StandardScaler()),
                ("lda", LDA(n_components=None)),
            ]),
            features,
        )
    ],
)
lda_gbm = Pipeline([
    ("preprocess", lda_preprocess),
    ("gb", GradientBoostingClassifier(random_state=RANDOM_SEED)),
])
lda_gbm.fit(X, y_enc)
y_pred = lda_gbm.predict(X_test)
print(f"LDA+GBM — Test accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


## LdaBoost on external test


In [ ]:
X_np = X.to_numpy()
X_test_np = X_test.to_numpy()

ldaboost = LdaBoost(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=RANDOM_SEED)
ldaboost.fit(X_np, y_enc)
y_pred = ldaboost.predict(X_test_np)

print(f"LdaBoost — Test accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


## Notes

- All steps use a fixed seed via `RANDOM_SEED = 42`.
- Imports of the algorithm use `from LdaBoost import LdaBoost` from the local package.
- Column names are sanitized to avoid issues with special characters.
